In [145]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import re
import nltk
from nltk.corpus import stopwords 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
import os
import joblib
import streamlit as st 

In [146]:
# reading data
data = pd.read_csv("resume_dataset_2.csv")
data.head()

,Name,Email,Phone,University,Graduation_Year,Years_Experience,Job_Role,Skills,Resume_Text
0,Tara Gonzalez,deborah75@example.com,8.371518e+09,Jadavpur University,2018,3,Data Scientist,"Python, Machine Learning, NumPy, Scikit-learn,...",\n Name: Tara Gonzalez\n Email: ...
1,Jared Quinn,matthew65@example.com,4.016587e+09,Delhi University,2017,6,Data Scientist,"NumPy, Scikit-learn, Statistics, TensorFlow, M...",\n Name: Jared Quinn\n Email: ma...
2,Jane Woods,phyllis29@example.net,9.002601e+09,Mumbai University,2020,5,Data Scientist,"TensorFlow, Python, NumPy, Machine Learning, NLP",\n Name: Jane Woods\n Email: phy...
3,Julie Chambers,hwright@example.net,4.157821e+09,Jadavpur University,2017,0,Data Scientist,"Scikit-learn, Statistics, Data Visualization, ...",\n Name: Julie Chambers\n Email:...
4,Tonya Campbell,hmyers@example.com,1.087373e+09,BITS Pilani,2015,2,Data Scientist,"Python, NumPy, SQL, Statistics, Pandas",\n Name: Tonya Campbell\n Email:...


In [147]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815 entries, 0 to 814
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Name              815 non-null    object 
 1   Email             779 non-null    object 
 2   Phone             779 non-null    float64
 3   University        779 non-null    object 
 4   Graduation_Year   815 non-null    int64  
 5   Years_Experience  815 non-null    int64  
 6   Job_Role          815 non-null    object 
 7   Skills            779 non-null    object 
 8   Resume_Text       815 non-null    object 
dtypes: float64(1), int64(2), object(6)
memory usage: 57.4+ KB


In [148]:
data.describe()

,Phone,Graduation_Year,Years_Experience
count,7.790000e+02,815.000000,815.000000
mean,4.815580e+09,2019.001227,4.089571
std,2.887972e+09,2.659413,2.442071
min,3.937791e+07,2015.000000,0.000000
25%,2.363896e+09,2017.000000,2.000000
50%,4.604395e+09,2019.000000,4.000000
75%,7.318987e+09,2021.000000,6.000000
max,9.990742e+09,2023.000000,8.000000


In [149]:
data.columns

Index(['Name', 'Email', 'Phone', 'University', 'Graduation_Year',
       'Years_Experience', 'Job_Role', 'Skills', 'Resume_Text'],
      dtype='object')

In [150]:
df = data[['Resume_Text', 'Skills', 'Years_Experience', 'Job_Role']]

In [151]:
data['combined_text'] = data['Resume_Text'] + " " + data['Skills']

In [152]:
data['Resume_Text'].value_counts()

Resume_Text
\n        Name: Natasha Turner\n        Email: jessicasellers@example.org\n        Phone: 5841465361\n\n        Education:\n        Bachelor's Degree from Anna University, 2015\n\n        Work Experience:\n        4 years of experience working as a Software Engineer.\n\n        Skills:\n        REST API, Kubernetes, Git, C++, Docker\n\n        Projects:\n        Worked on multiple industry-relevant projects demonstrating expertise in REST API, Kubernetes, Git.\n                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [153]:
# preprocessing data
stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

data['Cleaned'] = data['Resume_Text'].apply(clean_text)


C:\Users\Shubham\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\nltk\corpus\reader\api.py:218: RuntimeWarning: Security Violation [CorpusReader]: Unauthorized path C:\Users\Shubham\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\Roaming\nltk_data\corpora\stopwords\english
  with self.open(f) as fp:
C:\Users\Shubham\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\nltk\data.py:388: RuntimeWarning: Security Violation [pathsec.open]: Unauthorized path C:\Users\Shubham\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\Roaming\nltk_data\corpora\stopwords\english
  stream = _secure_open(self._path, "rb")


In [154]:
data.head()

,Name,Email,Phone,University,Graduation_Year,Years_Experience,Job_Role,Skills,Resume_Text,combined_text,Cleaned
0,Tara Gonzalez,deborah75@example.com,8.371518e+09,Jadavpur University,2018,3,Data Scientist,"Python, Machine Learning, NumPy, Scikit-learn,...",\n Name: Tara Gonzalez\n Email: ...,\n Name: Tara Gonzalez\n Email: ...,name tara gonzalez email deborah example com p...
1,Jared Quinn,matthew65@example.com,4.016587e+09,Delhi University,2017,6,Data Scientist,"NumPy, Scikit-learn, Statistics, TensorFlow, M...",\n Name: Jared Quinn\n Email: ma...,\n Name: Jared Quinn\n Email: ma...,name jared quinn email matthew example com pho...
2,Jane Woods,phyllis29@example.net,9.002601e+09,Mumbai University,2020,5,Data Scientist,"TensorFlow, Python, NumPy, Machine Learning, NLP",\n Name: Jane Woods\n Email: phy...,\n Name: Jane Woods\n Email: phy...,name jane woods email phyllis example net phon...
3,Julie Chambers,hwright@example.net,4.157821e+09,Jadavpur University,2017,0,Data Scientist,"Scikit-learn, Statistics, Data Visualization, ...",\n Name: Julie Chambers\n Email:...,\n Name: Julie Chambers\n Email:...,name julie chambers email hwright example net ...
4,Tonya Campbell,hmyers@example.com,1.087373e+09,BITS Pilani,2015,2,Data Scientist,"Python, NumPy, SQL, Statistics, Pandas",\n Name: Tonya Campbell\n Email:...,\n Name: Tonya Campbell\n Email:...,name tonya campbell email hmyers example com p...


In [155]:
data['Cleaned'].value_counts()

Cleaned
name linda stone email glenn example org phone education bachelor degree iim ahmedabad work experience years experience working software engineer skills                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               4
name natasha turner email jessicasellers example org phone education bachelor degree anna university work experience years experience working software engineer skills rest api kubernetes git c docker

In [156]:
tfidf = TfidfVectorizer()
text_features = tfidf.fit_transform(data['Cleaned']).toarray()


In [157]:
# Adding experiance feature 
exp = data['Years_Experience'].values.reshape(-1, 1)
X = np.hstack((text_features, exp))
Y = data['Job_Role'].values

In [158]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_mean=False)  # TF-IDF ke liye important
X = scaler.fit_transform(X)

In [159]:
# train-test split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [160]:
# KNN model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, Y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [161]:
# Naive Bayes 
nb = MultinomialNB()
nb.fit(X_train, Y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [162]:
svm = SVC(kernel='linear')
svm.fit(X_train, Y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [163]:
# Neural networks
nn = MLPClassifier(hidden_layer_sizes=(100,50), max_iter=500)
nn.fit(X_train, Y_train)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(100, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",500
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",None


In [164]:
model = {"KNN": knn, "Naive Bayes": nb, "SVM": svm, "Neural Network": nn}

for name , mdl in model.items():
    Y_pred = mdl.predict(X_test)
    print(f"{name} Accuracy:", accuracy_score(Y_test, Y_pred))
    print(f'{name} Classification Report :', classification_report(Y_test, Y_pred))
    print(f'{name} Confusion Matrix :', confusion_matrix(Y_test, Y_pred))

KNN Accuracy: 0.4171779141104294
KNN Classification Report :                    precision    recall  f1-score   support

   Data Scientist       0.00      0.00      0.00        30
Financial Analyst       0.00      0.00      0.00        16
     HR Executive       0.33      0.04      0.07        26
Marketing Manager       0.00      0.00      0.00        23
Software Engineer       0.42      0.99      0.59        68

         accuracy                           0.42       163
        macro avg       0.15      0.20      0.13       163
     weighted avg       0.23      0.42      0.26       163

KNN Confusion Matrix : [[ 0  0  1  0 29]
 [ 0  0  1  0 15]
 [ 0  0  1  0 25]
 [ 0  0  0  0 23]
 [ 0  0  0  1 67]]
Naive Bayes Accuracy: 0.7116564417177914
Naive Bayes Classification Report :                    precision    recall  f1-score   support

   Data Scientist       0.65      0.50      0.57        30
Financial Analyst       0.52      0.81      0.63        16
     HR Executive       0.67      0.

C:\Users\Shubham\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Shubham\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Shubham\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics

In [165]:
os.makedirs("models", exist_ok=True)
joblib.dump(knn, "models/knn.pkl")
joblib.dump(nb, "models/nb.pkl")
joblib.dump(svm, "models/svm.pkl")
joblib.dump(nn, "models/nn.pkl")
joblib.dump(tfidf, "models/tfidf.pkl")

['models/tfidf.pkl']

In [166]:
if 	data['Years_Experience'].iloc[0]> 2:
    print("Shortlisted")
else:
    print("Rejected")

Shortlisted


In [167]:
best_model = max(model.items(), key=lambda x: accuracy_score(Y_test, x[1].predict(X_test)))
print("Best Model:", best_model[0])

Best Model: Naive Bayes


In [168]:
# job_description = "Looking for a data scientist with experience in machine learning and Python."
# jb_vec = tfidf.transform([job_description])
# resume_vec = tfidf.transform([data['Cleaned'][0]])
# Score = cosine_similarity(jb_vec, resume_vec)
# print("Resume Score:", Score[0][0])

In [169]:
def get_top_candidates(jd, top_n=5):
    jd_vec = tfidf.transform([jd])
    resume_vecs = tfidf.transform(data['Cleaned'])
    
    scores = cosine_similarity(jd_vec, resume_vecs)[0]
    
    top_idx = scores.argsort()[-top_n:][::-1]
    
    results = data.iloc[top_idx][['Name', 'Job_Role', 'Skills', 'Years_Experience']]
    results['Match_Score'] = scores[top_idx]
    
    return results

In [170]:
jd = input("Enter Job Description: ")
results = get_top_candidates(jd)

for i, row in results.iterrows():
    print(f"""
Name: {row['Name']}
Role: {row['Job_Role']}
Experience: {row['Years_Experience']} years
Match Score: {row['Match_Score']:.2f}
Skills: {row['Skills']}
\n
""")
    
print(results)


Name: Jason Cox
Role: Financial Analyst
Experience: 2 years
Match Score: 0.00
Skills: Git, REST API, Spring Boot, Java, C++




Name: Bryan Chapman
Role: Software Engineer
Experience: 2 years
Match Score: 0.00
Skills: nan




Name: Stacy Smith
Role: Marketing Manager
Experience: 5 years
Match Score: 0.00
Skills: Java, C++, Spring Boot, OOP, REST API




Name: Ann White
Role: Software Engineer
Experience: 7 years
Match Score: 0.00
Skills: OOP, Microservices, Python, Git, Kubernetes




Name: Jeffrey Gonzalez
Role: Software Engineer
Experience: 5 years
Match Score: 0.00
Skills: C++, Microservices, Docker, Spring Boot, REST API



                 Name           Job_Role  \
814         Jason Cox  Financial Analyst   
813     Bryan Chapman  Software Engineer   
812       Stacy Smith  Marketing Manager   
811         Ann White  Software Engineer   
810  Jeffrey Gonzalez  Software Engineer   

                                                Skills  Years_Experience  \
814              Git, 

In [171]:
for i, row in results.iterrows():
    print(f"""
Name: {row['Name']}
Role: {row['Job_Role']}
Experience: {row['Years_Experience']} years
Match Score: {row['Match_Score']:.2f}
Skills: {row['Skills']}
---------------------------
""")


Name: Jason Cox
Role: Financial Analyst
Experience: 2 years
Match Score: 0.00
Skills: Git, REST API, Spring Boot, Java, C++
---------------------------


Name: Bryan Chapman
Role: Software Engineer
Experience: 2 years
Match Score: 0.00
Skills: nan
---------------------------


Name: Stacy Smith
Role: Marketing Manager
Experience: 5 years
Match Score: 0.00
Skills: Java, C++, Spring Boot, OOP, REST API
---------------------------


Name: Ann White
Role: Software Engineer
Experience: 7 years
Match Score: 0.00
Skills: OOP, Microservices, Python, Git, Kubernetes
---------------------------


Name: Jeffrey Gonzalez
Role: Software Engineer
Experience: 5 years
Match Score: 0.00
Skills: C++, Microservices, Docker, Spring Boot, REST API
---------------------------

